<a href="https://colab.research.google.com/github/aritrasinha531-ai/silver-invention/blob/main/FLAPPY_BIRD_GAME.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install flappy_bird_gymnasium
!pip install gymnasium
import torch
import torchvision
!apt-get install -y xvfb
!pip install pyvirtualdisplay


In [ ]:
import gymnasium as gym
import flappy_bird_gymnasium
import numpy as np
import torch
import torch.nn as nn
import random
from collections import deque
from pyvirtualdisplay import Display
from IPython.display import HTML
from base64 import b64encode
import torch.optim as optim
import cv2

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
display=Display(visible=0,size=(400,600))
display.start()

In [ ]:
EPISODES=300
BATCH_SIZE=64
GAMMA=0.99
LR=0.001
MEMORY_SIZE=10000
EPSILON_START=1.0
EPSILON_MIN=0.05
EPSILON_DECAY=0.995
TARGET_UPDATE=10
STATE_SIZE=180

In [ ]:
env=gym.make("FlappyBird-v0")
ACTION_SIZE=env.action_space.n
print("STATE_SIZE=",STATE_SIZE,"ACTION_SIZE=",ACTION_SIZE)

In [ ]:
# define DQN network
class DQN(nn.Module):
  def __init__(self,state_size,action_size):
    super(DQN,self). __init__()
    self.fc1=nn.Linear(state_size,128)
    self.fc2=nn.Linear(128,128)
    self.fc3=nn.Linear(128,action_size)
  def forward(self,x):
    x=torch.relu(self.fc1(x))
    x=torch.relu(self.fc2(x))
    return x
main_model=DQN(STATE_SIZE,ACTION_SIZE).to(device)
target_model=DQN(STATE_SIZE,ACTION_SIZE).to(device)
target_model.load_state_dict(main_model.state_dict())
optimizer=optim.Adam(main_model.parameters())
loss_fn=nn.MSELoss()

In [ ]:
# experience replay
memory=deque(maxlen=MEMORY_SIZE)
def store_experience(state,action,reward,next_state,done):
  memory.append((state,action,reward,next_state,done))
def sample_experience(batch_size):
  return random.sample(memory,batch_size)

In [ ]:
# Epsilon-greedy policy
epsilon=EPSILON_START
def choose_action(state):
  global epsilon
  if random.random()<epsilon:
    return random.randint(0,ACTION_SIZE-1)
    state_tensor=torch.FloatTensor(state).unsqueeze(0).to(device)
    with torch.no_grad():
      q_values=main_model(state_tensor)
    return torch.argmax(q_values).item()

In [ ]:
#training step
def train_step():
  if len(memory)<BATCH_SIZE:
    return
  batch=sample_experience(BATCH_SIZE)
  states,actions,rewards,next_states,dones=zip(*batch)
  states=torch.FloatTensor(states).to(device)
  actions=torch.LongTensor(actions).unsqueeze(1).to(device)
  rewards=torch.FloatTensor(rewards).to(device)
  next_states=torch.FloatTensor(next_states).to(device)
  dones=torch.FloatTensor(dones).to(device)
  q_values=main_model(states).gather(1,actions).squeeze(1)
  max_next_q=torch.max(target_model(next_states),dim=1)[0].detach()
  target=rewards+GAMMA*max_next_q*(1-dones)
  loss=loss_fn(q_values,target)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

In [ ]:
# training loop
for episode in range(EPISODES):
  state,_=env.reset()
  total_reward=0
  while True:
    action=choose_action(state)
    next_states,reward,terminated,truncated,_=env.step(action)
    dones=terminated or truncated
    store_experience(state,action,reward,next_states,dones)
    train_step()
    state=next_states
    total_reward+=reward
    if dones:
      break
# decay epsilon
EPSILON=max(EPSILON_MIN,epsilon*EPSILON_DECAY)
if episode%TARGET_UPDATE==0:
     target_model.load_state_dict(main_model.state_dict())
print(f"episode={episode+1} amd reward={total_reward}")


In [ ]:
env=gym.make("FlappyBird-v0", render_mode='rgb_array') # Re-initialize env with render_mode
state,_=env.reset()
dones=False
while not dones:
  state_tensor=torch.FloatTensor(state).unsqueeze(0).to(device)
  action=torch.argmax(main_model(state_tensor)).item()
  next_states,reward,dones,_,_=env.step(action)
  state=next_states
  frame=env.render()
  import time
  from IPython.display import clear_output
  import matplotlib.pyplot as plt
  clear_output(wait=True)
  plt.figure(figsize=(6,8))
  plt.imshow(frame)
  plt.axis('off')
  plt.show()

time.sleep(0.45)